In [1]:
# MNIST dataset으로 CNN 실습 - Sub classing
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터 준비
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# print(x_train[0])
print(x_train.shape) # (60000, 28, 28)

# 채널(channel) 추가 (흑백인 경우 1) -> 구조 변경후(3차원 -> 4차원 : CNN이 필요한 차원으로 변경) 정규화
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
print(x_train.shape) # (60000, 28, 28, 1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
(60000, 28, 28)
(60000, 28, 28, 1)


In [10]:
# 모델 정의
# 사용자 작성 class, 함수, 레이어 등을 keras모델 저장/리로딩시 자동으로 등록해주는 데코레이터
@tf.keras.utils.register_keras_serializable(package='custom') # 'custom', 'losses', 'activation'...
class MyMnistCnn(tf.keras.Model):
  def __init__(self, **kwargs):
    super().__init__(**kwargs)

    # Conv Block1
    self.conv1 = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')
    self.pool1 = tf.keras.layers.MaxPool2D(pool_size=(2,2))

    # Conv Block2
    self.conv2 = tf.keras.layers.Conv2D(filters=64, kernel_size=(3,3), padding='same',activation='relu')
    self.pool2 = tf.keras.layers.MaxPool2D(pool_size=(2,2))

    # FC layer(Fully Connected Layer) - 1차원으로 구조 변경
    self.falt = tf.keras.layers.Flatten()

    self.d1 = tf.keras.layers.Dense(units=64, activation='relu')
    self.drop1 = tf.keras.layers.Dropout(rate=0.2)
    self.d2 = tf.keras.layers.Dense(units=32, activation='relu')
    self.drop2 = tf.keras.layers.Dropout(rate=0.2)
    self.out = tf.keras.layers.Dense(units=10, activation='softmax')

  def call(self, inputs, training=False):
    x = self.conv1(inputs)
    x = self.pool1(x)
    x = self.conv2(inputs)
    x = self.pool2(x)
    x = self.falt(x)
    x = self.d1(x)
    x = self.drop1(x, training=training)
    x = self.d2(x)
    x = self.drop2(x, training=training)
    return self.out(x)

model = MyMnistCnn()
model.build(input_shape=(None, 28, 28, 1))
print(model.summary())



/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'my_mnist_cnn_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "my_mnist_cnn_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [11]:
# 3) compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
hist = model.fit(
    x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=1
)

# 4) 모델 평가하기 (train, test평가 점수의 차이가 크면 overfitting 의심)
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
print(f'train loss : {train_loss:.4f}, train_acc : {train_acc:.4f}')

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'test loss : {train_loss:.4f}, test_acc : {train_acc:.4f}')

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['my_mnist_cnn_4/conv2d_16/kernel', 'my_mnist_cnn_4/conv2d_16/bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.8688 - loss: 0.4214 - val_accuracy: 0.9795 - val_loss: 0.0762
Epoch 2/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9591 - loss: 0.1455 - val_accuracy: 0.9848 - val_loss: 0.0557
Epoch 3/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9716 - loss: 0.1017 - val_accuracy: 0.9860 - val_loss: 0.0485
Epoch 4/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9770 - loss: 0.0805 - val_accuracy: 0.9855 - val_loss: 0.0504
Epoch 5/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9804 - loss: 0.0664 - val_accuracy: 0.9878 - val_loss: 0.0442
Epoch 6/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9836 - loss: 0.0557 - val_accuracy: 0.9895 - val_loss: 0.0389
Epoch 7/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9858 - loss: 0.0485 - val_accuracy: 0.9892 - val_loss: 0.0435
Epoch 8/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9871 - loss: 0.0427 - val_accuracy: 0.98